## Supplementary Table — Parcelwise Structural Brain Differences

Exports all parcelwise T-statistics, Hedges' g, and FDR-corrected q-values to a formatted
Excel file for use as a supplementary table in the thesis.

**Phenotypes:**
- Cortical thickness (Desikan–Killiany atlas, 68 parcels per contrast)
- Subcortical volume (FreeSurfer aseg atlas, 14 bilateral structures per contrast)

**Contrasts:** *de novo* PD vs HC · prodromal PD vs HC · *de novo* PD vs prodromal PD

**Output:** `results/supplementary_parcelwise_results.xlsx`

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from statsmodels.stats.multitest import multipletests
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

RESULTS_DIR = Path("../../results")
OUT_PATH    = RESULTS_DIR / "supplementary_parcelwise_results.xlsx"

In [ ]:
# ── Group sizes per contrast ───────────────────────────────────────────────────
GROUP_N = {
    "de novo PD vs HC":           {"n1": 558, "n2": 192},
    "prodromal PD vs HC":         {"n1": 279, "n2": 192},
    "de novo PD vs prodromal PD": {"n1": 558, "n2": 279},
}

# ── Cortical thickness CSV paths ───────────────────────────────────────────────
THICK_FILES = {
    "de novo PD vs HC":           RESULTS_DIR / "de novo pd vs hc_parcelwise_ttest.csv",
    "prodromal PD vs HC":         RESULTS_DIR / "prodromal pd vs hc_parcelwise_ttest.csv",
    "de novo PD vs prodromal PD": RESULTS_DIR / "de novo pd vs prodromal pd_parcelwise_ttest.csv",
}

# ── Subcortical volume CSV paths ───────────────────────────────────────────────
SUBC_FILES = {
    "de novo PD vs HC":           RESULTS_DIR / "de_novo_pd_vs_hc_parcelwise_ttest_volume_subcortical.csv",
    "prodromal PD vs HC":         RESULTS_DIR / "prodromal_pd_vs_hc_parcelwise_ttest_volume_subcortical.csv",
    "de novo PD vs prodromal PD": RESULTS_DIR / "de_novo_pd_vs_prodromal_pd_parcelwise_ttest_volume_subcortical.csv",
}

In [ ]:
def hedges_g(t, n1, n2, df):
    """
    Bias-corrected Hedges' g from OLS T-statistic.
    Negated so that positive g = more in first-named group.
    """
    J = 1 - 3 / (4 * df - 1)
    return -t * np.sqrt(1/n1 + 1/n2) * J


def load_and_compute(files, group_n, fdr_n_label):
    """Load CSVs, compute Hedges' g, apply FDR within each contrast."""
    frames = []
    for contrast, path in files.items():
        df = pd.read_csv(path)
        df = df.dropna(subset=["pvalue"])
        n1 = group_n[contrast]["n1"]
        n2 = group_n[contrast]["n2"]
        df["Hedges_g"] = hedges_g(df["Tvalue"], n1, n2, df["df"])
        df["contrast"] = contrast
        df["n1"] = n1
        df["n2"] = n2
        # FDR within contrast
        _, q_vals, _, _ = multipletests(df["pvalue"], method="fdr_bh")
        df["q_FDR"] = q_vals
        frames.append(df)
    return pd.concat(frames, ignore_index=True)


df_thick = load_and_compute(THICK_FILES, GROUP_N, "68 parcels")
df_subc  = load_and_compute(SUBC_FILES,  GROUP_N, "14 structures")

print(f"Cortical thickness: {len(df_thick)} rows")
print(f"Subcortical volume: {len(df_subc)} rows")

In [ ]:
CONTRAST_ORDER = [
    "de novo PD vs HC",
    "prodromal PD vs HC",
    "de novo PD vs prodromal PD",
]


def build_export_df(df):
    """
    Build a clean export DataFrame with human-readable columns.
    Sorted by contrast order then |Hedges' g| descending.
    """
    df = df.copy()
    df["Region"] = df["parcel"].str.replace(r".*_lab-", "", regex=True)
    df["Hemisphere"] = df["hemi"].map({"L": "Left", "R": "Right"})
    df["Significant (q < 0.05)"] = df["q_FDR"].apply(lambda q: "*" if q < 0.05 else "")
    df["abs_g"] = df["Hedges_g"].abs()
    df["contrast_order"] = df["contrast"].map(
        {c: i for i, c in enumerate(CONTRAST_ORDER)}
    )

    out = df.sort_values(["contrast_order", "abs_g"], ascending=[True, False])

    cols = [
        "Region", "Hemisphere", "contrast", "n1", "n2",
        "Tvalue", "df", "pvalue", "Hedges_g", "q_FDR", "Significant (q < 0.05)",
    ]
    rename = {
        "contrast":  "Contrast",
        "n1":        "N (group 1)",
        "n2":        "N (group 2)",
        "Tvalue":    "T-statistic",
        "df":        "df",
        "pvalue":    "p (uncorrected)",
        "Hedges_g":  "Hedges' g",
        "q_FDR":     "q (FDR)",
    }
    return out[cols].rename(columns=rename).reset_index(drop=True)


df_thick_export = build_export_df(df_thick)
df_subc_export  = build_export_df(df_subc)

print("Cortical thickness export:")
print(df_thick_export.head(8).to_string())
print("\nFDR-significant rows:", (df_thick_export["Significant (q < 0.05)"] == "*").sum())

In [ ]:
# ── Excel formatting helpers ──────────────────────────────────────────────────

HEADER_FILL   = PatternFill("solid", fgColor="2B5590")   # dark blue
HEADER_FONT   = Font(bold=True, color="FFFFFF", size=10)
SIG_FILL      = PatternFill("solid", fgColor="FFFFE0")   # light yellow
NORMAL_FONT   = Font(size=10)
CENTER_ALIGN  = Alignment(horizontal="center", vertical="center", wrap_text=False)
LEFT_ALIGN    = Alignment(horizontal="left",   vertical="center")

THIN_BORDER_SIDE = Side(style="thin", color="CCCCCC")
THIN_BORDER      = Border(
    left=THIN_BORDER_SIDE, right=THIN_BORDER_SIDE,
    top=THIN_BORDER_SIDE,  bottom=THIN_BORDER_SIDE,
)

# Numeric format strings
FMT_2DP  = "0.00"
FMT_4DP  = "0.0000"
FMT_INT  = "0"

COL_FORMATS = {
    "T-statistic":      FMT_4DP,
    "df":               FMT_2DP,
    "p (uncorrected)":  FMT_4DP,
    "Hedges' g":        FMT_4DP,
    "q (FDR)":          FMT_4DP,
    "N (group 1)":      FMT_INT,
    "N (group 2)":      FMT_INT,
}


def write_sheet(ws, df, sheet_title):
    """Write a DataFrame to an openpyxl worksheet with formatting."""
    headers = list(df.columns)

    # Header row
    for col_idx, header in enumerate(headers, 1):
        cell = ws.cell(row=1, column=col_idx, value=header)
        cell.fill      = HEADER_FILL
        cell.font      = HEADER_FONT
        cell.alignment = CENTER_ALIGN
        cell.border    = THIN_BORDER

    # Data rows
    for row_idx, (_, row) in enumerate(df.iterrows(), 2):
        is_sig = row.get("Significant (q < 0.05)") == "*"
        for col_idx, (col_name, value) in enumerate(zip(headers, row), 1):
            cell = ws.cell(row=row_idx, column=col_idx, value=value)
            cell.font   = NORMAL_FONT
            cell.border = THIN_BORDER
            if is_sig:
                cell.fill = SIG_FILL
            # Numeric format
            if col_name in COL_FORMATS and isinstance(value, (int, float)):
                cell.number_format = COL_FORMATS[col_name]
            # Alignment
            if col_name in ("Region", "Hemisphere", "Contrast"):
                cell.alignment = LEFT_ALIGN
            else:
                cell.alignment = CENTER_ALIGN

    # Auto-width columns
    for col_idx, col_name in enumerate(headers, 1):
        col_values = [str(col_name)] + [
            str(v) for v in df.iloc[:, col_idx - 1]
        ]
        max_width = min(max(len(v) for v in col_values) + 2, 30)
        ws.column_dimensions[get_column_letter(col_idx)].width = max_width

    # Freeze top row
    ws.freeze_panes = "A2"

    # Row height for header
    ws.row_dimensions[1].height = 20

    print(f"  Sheet '{ws.title}': {len(df)} rows, {is_sig_count(df)} FDR-significant")


def is_sig_count(df):
    return (df["Significant (q < 0.05)"] == "*").sum()


# ── Build workbook ────────────────────────────────────────────────────────────
wb = openpyxl.Workbook()

# Sheet 1: Cortical Thickness
ws1 = wb.active
ws1.title = "Cortical Thickness"
write_sheet(ws1, df_thick_export, "Cortical Thickness")

# Sheet 2: Subcortical Volume
ws2 = wb.create_sheet(title="Subcortical Volume")
write_sheet(ws2, df_subc_export, "Subcortical Volume")

wb.save(OUT_PATH)
print(f"\nSaved: {OUT_PATH}")

## Notes

**Hedges' g formula:** `g = −T × √(1/n₁ + 1/n₂) × J`, where `J = 1 − 3/(4×df − 1)` is the small-sample bias correction factor. The negation ensures positive g = greater value in the first-named group.

**Sign convention:**
- For "*de novo* PD vs HC": g > 0 = *de novo* PD has more thickness / volume than HC.
- For "prodromal PD vs HC": g > 0 = prodromal PD has more thickness / volume than HC.

**FDR correction:** Benjamini–Hochberg, applied within each contrast separately (68 tests for cortical thickness, 14 tests for subcortical volume).

**Covariates in OLS model:** Cortical thickness — age, sex, MRI field strength. Subcortical volume — age, sex, estimated total intracranial volume (eTIV), MRI field strength.

**Light yellow rows** = FDR-significant (q < 0.05). Asterisk (*) in the "Significant" column.